In [1]:
# 06_explain_shap.ipynb
# Post-hoc Explainability using SHAP

import sys
import time
import psutil
import os
import tracemalloc
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_INSTANCES


import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import shap
from sklearn.model_selection import StratifiedShuffleSplit

processed_path = Path("../datasets/processed")
models_path = Path("../results/black_box_models")
results_path = Path("../results/SHAP")
results_path.mkdir(parents=True, exist_ok=True)

CLASS_IDX = 1

for ds in DATASETS:
    print(f"\nDATASET: {ds}")

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    s = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    train_idx, test_idx = next(s.split(X, y))
    X_train = X.iloc[train_idx].reset_index(drop=True)
    X_test  = X.iloc[test_idx].reset_index(drop=True)
    y_test  = y.iloc[test_idx].reset_index(drop=True)

    defect_idx = np.where(y_test.values == 1)[0]
    if len(defect_idx) == 0:
        print(f"  No defective instances in {ds}")
        continue
    selected_idx = defect_idx[:N_INSTANCES]

    dataset_model_path = models_path / ds

    for model_dir in sorted(dataset_model_path.iterdir()):
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f" → {model_name}")

        model = joblib.load(model_dir / "model.joblib")

        save_path = results_path / ds / model_name
        save_path.mkdir(parents=True, exist_ok=True)

        is_tree = any(x in model_name.lower() for x in ["random", "gradient", "forest", "boost"])

        if is_tree:
            explainer = shap.TreeExplainer(model)
        else:
            background = X_train.sample(
                n=min(100, len(X_train)),
                random_state=RANDOM_STATE
            ).values
            explainer = shap.KernelExplainer(
                lambda x: model.predict_proba(x),
                background
            )

        records = []
        shap_times = []

        _mem_before_loop = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        tracemalloc.start()

        for i, idx in enumerate(selected_idx):
            instance = X_test.iloc[idx:idx + 1].values

            _t0 = time.perf_counter()
            shap_values = explainer.shap_values(instance)
            shap_times.append(time.perf_counter() - _t0)

            if isinstance(shap_values, list):
                shap_values = shap_values[CLASS_IDX]

            shap_vector = shap_values[0].tolist()
        
        _, _peak_traced = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        _mem_after_loop = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
        _peak_mb = max(_mem_after_loop - _mem_before_loop, _peak_traced / (1024 * 1024))

        timing_entry = {
            "Dataset": ds, "Model": model_name, "Method": "SHAP",
            "Explainer_Type": "TreeExplainer" if is_tree else "KernelExplainer",
            "Explain_Time_Mean_s": round(float(sum(shap_times)/len(shap_times)), 4),
            "Explain_Time_Std_s":  round(float(pd.Series(shap_times).std()), 4),
            "Mem_Delta_MB": round(_peak_mb, 2)
        }
        timing_csv = results_path / ds / model_name / "shap_timing.csv"
        pd.DataFrame([timing_entry]).to_csv(timing_csv, index=False)
        print(f"   SHAP time: {timing_entry['Explain_Time_Mean_s']:.4f}s ± {timing_entry['Explain_Time_Std_s']:.4f}s | Peak Mem: {timing_entry['Mem_Delta_MB']:.1f} MB")


DATASET: CM1
 → Gradient_Boosting
   SHAP time: 0.0009s ± 0.0002s | Peak Mem: 0.1 MB
 → MLP


/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/xai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


   SHAP time: 35.7147s ± 3.4235s | Peak Mem: 74.6 MB
